# OCR Implementation untuk Aksara Jawa

Pipeline ini mencakup:
1. Load Pre-trained Model
2. Text Detection & Segmentation
3. Character Classification
4. OCR Result Visualization

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import timm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2
from PIL import Image
from pathlib import Path

from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Pre-trained Model

In [ ]:
IMG_SIZE = 224

class_names = ['ba', 'ca', 'da', 'dha', 'ga', 'ha', 'ja', 'ka', 'la', 'ma', 
               'na', 'nga', 'nya', 'pa', 'ra', 'sa', 'ta', 'tha', 'wa', 'ya']

label_encoder = LabelEncoder()
label_encoder.fit(class_names)
num_classes = len(class_names)

print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=False, num_classes=num_classes)
model.load_state_dict(torch.load('best_model.pth', map_location=device))
model = model.to(device)
model.eval()

print("Model loaded successfully")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 2. Text Detection & Segmentation

In [ ]:
def preprocess_image(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)
    return gray

def detect_characters(image_path, min_area=100, max_area=50000):
    img = cv2.imread(str(image_path))
    original = img.copy()
    
    gray = preprocess_image(img)
    
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel, iterations=1)
    
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    bounding_boxes = []
    for contour in contours:
        area = cv2.contourArea(contour)
        if min_area < area < max_area:
            x, y, w, h = cv2.boundingRect(contour)
            bounding_boxes.append((x, y, w, h))
    
    bounding_boxes = sorted(bounding_boxes, key=lambda b: (b[1], b[0]))
    
    return original, gray, bounding_boxes

print("Detection functions defined")

## 3. Character Classification

In [ ]:
def classify_character(img_crop, model, transform, device):
    img_crop = cv2.cvtColor(img_crop, cv2.COLOR_BGR2GRAY)
    img_crop = cv2.equalizeHist(img_crop)
    img_crop = cv2.GaussianBlur(img_crop, (3, 3), 0)
    img_crop = cv2.cvtColor(img_crop, cv2.COLOR_GRAY2RGB)
    
    img_pil = Image.fromarray(img_crop)
    img_tensor = transform(img_pil).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(img_tensor)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = torch.max(probabilities, 1)
    
    predicted_class = label_encoder.inverse_transform([predicted.item()])[0]
    confidence_score = confidence.item()
    
    return predicted_class, confidence_score

print("Classification function defined")

## 4. OCR Pipeline

In [ ]:
def ocr_aksara_jawa(image_path, model, transform, device, confidence_threshold=0.5):
    original, gray, bounding_boxes = detect_characters(image_path)
    
    results = []
    for i, (x, y, w, h) in enumerate(bounding_boxes):
        char_crop = original[y:y+h, x:x+w]
        
        if char_crop.size == 0:
            continue
        
        predicted_class, confidence = classify_character(char_crop, model, transform, device)
        
        if confidence >= confidence_threshold:
            results.append({
                'index': i,
                'bbox': (x, y, w, h),
                'character': predicted_class,
                'confidence': confidence
            })
    
    return original, results

print("OCR pipeline defined")

## 5. Visualization

In [ ]:
def visualize_results(image, results):
    fig, ax = plt.subplots(1, figsize=(16, 12))
    ax.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    
    for result in results:
        x, y, w, h = result['bbox']
        character = result['character']
        confidence = result['confidence']
        
        rect = patches.Rectangle((x, y), w, h, linewidth=2, 
                                 edgecolor='red', facecolor='none')
        ax.add_patch(rect)
        
        ax.text(x, y-5, f"{character} ({confidence:.2f})", 
               bbox=dict(facecolor='yellow', alpha=0.7), 
               fontsize=10, color='black')
    
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    text_sequence = ' '.join([r['character'] for r in results])
    print(f"\nDetected sequence: {text_sequence}")
    print(f"\nTotal characters detected: {len(results)}")
    print(f"Average confidence: {np.mean([r['confidence'] for r in results]):.2f}")

print("Visualization function defined")

## 6. Run OCR

In [ ]:
image_path = 'a9fecd1ed6ed2e5de191a6bc8f1b5e5d.jpg'

print("Running OCR on aksara Jawa document...")
original_image, results = ocr_aksara_jawa(image_path, model, transform, device, confidence_threshold=0.5)

print(f"\nDetection completed: {len(results)} characters found")

In [ ]:
visualize_results(original_image, results)

In [ ]:
df_results = pd.DataFrame(results)
print("\nDetailed Results:")
print(df_results[['index', 'character', 'confidence']].to_string(index=False))